# 05 — Data Cleaning dan Text Preprocessing Komentar YouTube

Notebook ini digunakan untuk melakukan proses data cleaning dan text preprocessing terhadap dataset raw komentar YouTube yang telah dikumpulkan pada tahap sebelumnya.

Fokus utama notebook ini adalah membersihkan teks komentar agar lebih siap digunakan pada tahap analisis sentimen berikutnya, tanpa mengubah dataset raw asli.

Tahapan ini tidak mencakup labeling sentimen, feature extraction, TF-IDF, modeling, evaluasi model, maupun deployment Streamlit.

## 1. Tujuan Tahap

Tujuan dari tahap ini adalah menyiapkan dataset komentar YouTube yang lebih bersih, aman, dan siap digunakan untuk tahap analisis sentimen berikutnya.

Secara khusus, tahap ini bertujuan untuk:

1. Membaca dataset raw komentar YouTube dari folder `data/raw/`.
2. Memastikan dataset raw tidak diubah secara langsung.
3. Memvalidasi keberadaan kolom utama `text_original`.
4. Membuat working DataFrame untuk proses cleaning.
5. Mengurangi risiko penyebaran data sensitif, terutama informasi author.
6. Menyiapkan struktur penyimpanan hasil cleaning pada folder `data/processed/`.
7. Menyimpan ringkasan proses cleaning ke folder `reports/`.

Tahap ini merupakan bagian dari proses Data Preparation dalam kerangka CRISP-DM, karena data mentah perlu dibersihkan terlebih dahulu sebelum dapat digunakan untuk analisis sentimen.

## 2. Output yang Dihasilkan

Output yang diharapkan dari notebook ini adalah:

1. Dataset hasil cleaning dalam format CSV yang disimpan ke folder:

   `data/processed/`

2. Ringkasan proses cleaning dalam format CSV yang disimpan ke folder:

   `reports/`

3. Informasi perbandingan jumlah data sebelum dan sesudah proses cleaning.

4. Dataset processed yang lebih aman dibandingkan dataset raw, terutama dengan menghapus atau menyamarkan kolom yang berpotensi sensitif seperti `author`.

Catatan penting:

- Dataset raw pada folder `data/raw/` tidak boleh diubah.
- File `.env` tidak boleh ditampilkan dan tidak boleh masuk ke GitHub.
- Kolom `author` tidak boleh ditampilkan secara berlebihan.
- Dataset processed tetap perlu diperiksa kembali sebelum diputuskan untuk diikutsertakan ke GitHub.

## 3. Langkah Teknis Bagian Awal

Pada bagian awal notebook ini, langkah teknis yang dilakukan adalah:

1. Import library yang dibutuhkan.
2. Mendeteksi lokasi project root.
3. Mendefinisikan path folder utama project.
4. Membuat folder `data/processed/` dan `reports/` jika belum tersedia.
5. Mendeteksi file dataset raw terbaru dari folder `data/raw/`.
6. Membaca dataset raw ke dalam DataFrame.
7. Menampilkan informasi awal dataset secara aman.
8. Memvalidasi keberadaan kolom utama `text_original`.

In [1]:
# ============================================================
# 05 - Data Cleaning dan Text Preprocessing Komentar YouTube
# Bagian 1: Import Library
# ============================================================

import os
import re
import html
import json
import unicodedata
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

# Konfigurasi tampilan pandas agar output lebih rapi
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 120)

print("Library berhasil di-import.")
print("Pandas version:", pd.__version__)

Library berhasil di-import.
Pandas version: 3.0.3


In [3]:
# ============================================================
# Deteksi Project Root
# ============================================================

current_path = Path.cwd()

# Jika notebook dijalankan dari folder notebooks, maka root adalah parent-nya.
# Jika notebook dijalankan dari root project, maka root tetap current_path.
if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

print("Current working directory:")
print(current_path)

print("\nProject root terdeteksi:")
print(project_root)

Current working directory:
D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis\notebooks

Project root terdeteksi:
D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis


In [4]:
# ============================================================
# Definisi Path Folder Project
# ============================================================

raw_data_dir = project_root / "data" / "raw"
processed_data_dir = project_root / "data" / "processed"
reports_dir = project_root / "reports"

# Membuat folder output jika belum tersedia
processed_data_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

print("Folder data raw:")
print(raw_data_dir)

print("\nFolder data processed:")
print(processed_data_dir)

print("\nFolder reports:")
print(reports_dir)

print("\nStatus folder:")
print("data/raw exists       :", raw_data_dir.exists())
print("data/processed exists :", processed_data_dir.exists())
print("reports exists        :", reports_dir.exists())

Folder data raw:
D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis\data\raw

Folder data processed:
D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis\data\processed

Folder reports:
D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis\reports

Status folder:
data/raw exists       : True
data/processed exists : True
reports exists        : True


In [5]:
# ============================================================
# Deteksi File Dataset Raw Terbaru
# ============================================================

# Ekstensi file dataset yang didukung
allowed_extensions = ["*.csv", "*.xlsx", "*.json"]

raw_files = []

for ext in allowed_extensions:
    raw_files.extend(list(raw_data_dir.glob(ext)))

if len(raw_files) == 0:
    raise FileNotFoundError(
        f"Tidak ditemukan file dataset raw di folder: {raw_data_dir}\n"
        "Pastikan file hasil crawling komentar YouTube sudah tersimpan di folder data/raw/."
    )

# Mengurutkan berdasarkan waktu modifikasi file, terbaru berada di urutan pertama
raw_files_sorted = sorted(raw_files, key=lambda x: x.stat().st_mtime, reverse=True)

latest_raw_file = raw_files_sorted[0]

print("Jumlah file dataset raw ditemukan:", len(raw_files_sorted))
print("File dataset raw terbaru:")
print(latest_raw_file.name)

print("\nWaktu modifikasi file terbaru:")
print(datetime.fromtimestamp(latest_raw_file.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S"))

Jumlah file dataset raw ditemukan: 2
File dataset raw terbaru:
youtube_comments_raw_20260529_141537.csv

Waktu modifikasi file terbaru:
2026-05-29 14:16:48


In [6]:
# ============================================================
# Load Dataset Raw Secara Aman
# ============================================================

file_suffix = latest_raw_file.suffix.lower()

if file_suffix == ".csv":
    df_raw = pd.read_csv(latest_raw_file)
elif file_suffix == ".xlsx":
    df_raw = pd.read_excel(latest_raw_file)
elif file_suffix == ".json":
    df_raw = pd.read_json(latest_raw_file)
else:
    raise ValueError(f"Format file belum didukung: {file_suffix}")

print("Dataset raw berhasil dibaca.")
print("Nama file:", latest_raw_file.name)
print("Jumlah baris:", df_raw.shape[0])
print("Jumlah kolom:", df_raw.shape[1])

Dataset raw berhasil dibaca.
Nama file: youtube_comments_raw_20260529_141537.csv
Jumlah baris: 3742
Jumlah kolom: 13


In [7]:
# ============================================================
# Cek Struktur Kolom Dataset Raw
# ============================================================

columns_info = pd.DataFrame({
    "column_name": df_raw.columns,
    "data_type": [df_raw[col].dtype for col in df_raw.columns],
    "non_null_count": [df_raw[col].notna().sum() for col in df_raw.columns],
    "null_count": [df_raw[col].isna().sum() for col in df_raw.columns],
    "null_percentage": [
        round((df_raw[col].isna().sum() / len(df_raw)) * 100, 2) if len(df_raw) > 0 else 0
        for col in df_raw.columns
    ]
})

columns_info

,column_name,data_type,non_null_count,null_count,null_percentage
0,video_id,str,3742,0,0.0
1,video_title,str,3742,0,0.0
2,video_url,str,3742,0,0.0
3,channel_title,str,3742,0,0.0
4,video_published_at,str,3742,0,0.0
5,comment_id,str,3742,0,0.0
6,author,str,3742,0,0.0
7,published_at,str,3742,0,0.0
8,comment_updated_at,str,3742,0,0.0
9,text_original,str,3742,0,0.0


In [8]:
# ============================================================
# Validasi Kolom Utama text_original
# ============================================================

required_columns = ["text_original"]

missing_columns = [col for col in required_columns if col not in df_raw.columns]

if missing_columns:
    raise KeyError(
        f"Kolom wajib tidak ditemukan: {missing_columns}\n"
        "Pastikan dataset raw memiliki kolom text_original dari hasil crawling komentar YouTube."
    )

print("Validasi berhasil.")
print("Kolom wajib tersedia:", required_columns)

Validasi berhasil.
Kolom wajib tersedia: ['text_original']


In [9]:
# ============================================================
# Preview Aman Dataset Raw
# ============================================================

def mask_author(value):
    """
    Fungsi untuk menyamarkan nama author agar tidak tampil secara utuh.
    """
    if pd.isna(value):
        return np.nan
    
    value = str(value)
    
    if len(value) <= 2:
        return "*" * len(value)
    
    return value[:2] + "***" + value[-1]


df_preview_safe = df_raw.copy()

# Masking kolom author jika tersedia
if "author" in df_preview_safe.columns:
    df_preview_safe["author"] = df_preview_safe["author"].apply(mask_author)

# Menentukan kolom yang aman untuk preview
safe_preview_columns = []

for col in ["comment_id", "video_id", "published_at", "like_count", "reply_count", "author", "text_original"]:
    if col in df_preview_safe.columns:
        safe_preview_columns.append(col)

df_preview_safe[safe_preview_columns].head(5)

,comment_id,video_id,published_at,like_count,reply_count,author,text_original
0,Ugzo_OAorVrtNPFL3cF4AaABAg,sB7fSPbYFJg,2026-05-24 01:37:15+00:00,560,11,@b***y,"gue kira di upload 10 jam yg lalu,ternyata 10 tahun yg lalu😂"
1,Ugx2WJ9HFPUNRPaOxON4AaABAg,sB7fSPbYFJg,2026-05-22 13:31:00+00:00,570,12,@I***l,"Timing nya gabisa lebih pas dari ini😭🙏\nNyentuh Rp.25.000,00 Masih Aman Indonesia😎👌"
2,UgwyjT6JiPsbjsMoUqR4AaABAg,sB7fSPbYFJg,2026-05-24 09:42:48+00:00,271,3,@b***i,Algoritma YouTube cukup mengerikan 🥀
3,UgwN7TGK6eU5papwnNZ4AaABAg,sB7fSPbYFJg,2026-05-21 06:57:48+00:00,321,2,@j***8,dyeem 10 year ago lewat beranda kuhh pas bgett;(((
4,UgzSaWu-A-vsRbjSc6B4AaABAg,sB7fSPbYFJg,2026-05-23 17:47:36+00:00,92,1,@G***r,Algoritma YouTube lebih mengerikan daripada Pria Solo☠️


In [10]:
# ============================================================
# Ringkasan Awal Dataset Raw
# ============================================================

initial_summary = pd.DataFrame({
    "metric": [
        "source_file",
        "total_rows_raw",
        "total_columns_raw",
        "text_original_null_count",
        "text_original_null_percentage",
        "duplicate_comment_id_count",
        "duplicate_text_original_count"
    ],
    "value": [
        latest_raw_file.name,
        df_raw.shape[0],
        df_raw.shape[1],
        df_raw["text_original"].isna().sum(),
        round((df_raw["text_original"].isna().sum() / len(df_raw)) * 100, 2) if len(df_raw) > 0 else 0,
        df_raw["comment_id"].duplicated().sum() if "comment_id" in df_raw.columns else "comment_id column not found",
        df_raw["text_original"].duplicated().sum()
    ]
})

initial_summary

,metric,value
0,source_file,youtube_comments_raw_20260529_141537.csv
1,total_rows_raw,3742
2,total_columns_raw,13
3,text_original_null_count,0
4,text_original_null_percentage,0.0
5,duplicate_comment_id_count,0
6,duplicate_text_original_count,43


## 4. Membuat Working DataFrame

Pada bagian ini, dataset raw tidak akan diubah secara langsung. Proses cleaning dilakukan pada salinan DataFrame baru yang disebut `df_work`.

Prinsip yang digunakan:

1. Dataset raw tetap dipertahankan apa adanya.
2. Kolom `author` tidak digunakan dalam dataset processed untuk mengurangi risiko penyebaran data sensitif.
3. Kolom `text_original` tetap digunakan sebagai sumber utama pembuatan kolom `text_clean`.
4. Proses cleaning dilakukan secara bertahap agar jumlah data yang dihapus dapat diaudit.

In [11]:
# ============================================================
# Membuat Working DataFrame
# ============================================================

df_work = df_raw.copy(deep=True)

# Menambahkan informasi nama file sumber untuk kebutuhan audit internal
df_work["source_file"] = latest_raw_file.name

# Menghapus kolom author dari working dataframe jika tersedia
# Tujuannya agar dataset processed tidak menyimpan nama author secara langsung
if "author" in df_work.columns:
    df_work = df_work.drop(columns=["author"])

print("Working DataFrame berhasil dibuat.")
print("Jumlah baris df_work:", df_work.shape[0])
print("Jumlah kolom df_work:", df_work.shape[1])

print("\nKolom pada df_work:")
print(df_work.columns.tolist())

Working DataFrame berhasil dibuat.
Jumlah baris df_work: 3742
Jumlah kolom df_work: 13

Kolom pada df_work:
['video_id', 'video_title', 'video_url', 'channel_title', 'video_published_at', 'comment_id', 'published_at', 'comment_updated_at', 'text_original', 'like_count', 'reply_count', 'crawl_timestamp', 'source_file']


## 5. Strategi Cleaning Teks

Proses cleaning teks pada tahap ini dilakukan untuk membersihkan komentar YouTube tanpa mengubah makna utama komentar secara berlebihan.

Tahapan cleaning yang dilakukan meliputi:

1. Membersihkan HTML entity.
2. Mengubah teks menjadi huruf kecil.
3. Menghapus URL.
4. Menghapus mention dan hashtag.
5. Menghapus newline, tab, dan spasi berlebih.
6. Menghapus emoji dan simbol yang tidak relevan secara aman.
7. Menghapus karakter non-alfanumerik yang tidak relevan.
8. Melakukan normalisasi kata tidak baku sederhana.
9. Membuat kolom baru `text_clean`.

Catatan:

- Pada tahap ini belum dilakukan stemming.
- Pada tahap ini belum dilakukan stopword removal.
- Pada tahap ini belum dilakukan TF-IDF.
- Pada tahap ini belum dilakukan labeling sentimen.
- Pada tahap ini belum dilakukan modeling.

In [13]:
# ============================================================
# Fungsi Statistik Teks
# ============================================================

def count_characters(text):
    """
    Menghitung jumlah karakter pada teks.
    """
    if pd.isna(text):
        return 0
    return len(str(text))


def count_words(text):
    """
    Menghitung jumlah kata berdasarkan pemisah spasi.
    """
    if pd.isna(text):
        return 0
    
    text = str(text).strip()
    
    if text == "":
        return 0
    
    return len(text.split())


print("Fungsi statistik teks berhasil dibuat.")

Fungsi statistik teks berhasil dibuat.


In [14]:
# ============================================================
# Normalisasi Kata Tidak Baku Sederhana
# ============================================================

normalization_dict = {
    "yg": "yang",
    "yng": "yang",
    "dgn": "dengan",
    "dg": "dengan",
    "utk": "untuk",
    "tdk": "tidak",
    "gak": "tidak",
    "ga": "tidak",
    "gk": "tidak",
    "nggak": "tidak",
    "ngga": "tidak",
    "enggak": "tidak",
    "krn": "karena",
    "karna": "karena",
    "tp": "tapi",
    "tpi": "tapi",
    "dr": "dari",
    "dlm": "dalam",
    "sdh": "sudah",
    "udah": "sudah",
    "blm": "belum",
    "bgt": "banget",
    "bangettt": "banget",
    "aja": "saja",
    "trs": "terus",
    "trus": "terus",
    "sm": "sama",
    "sama2": "sama sama",
    "gue": "saya",
    "gw": "saya",
    "gua": "saya",
    "lu": "kamu",
    "lo": "kamu",
    "loe": "kamu",
    "rp": "rupiah"
}


def normalize_slang_words(text):
    """
    Melakukan normalisasi kata tidak baku sederhana berdasarkan kamus manual.
    Kamus ini masih bersifat awal dan dapat dikembangkan pada tahap berikutnya.
    """
    if pd.isna(text):
        return ""
    
    tokens = str(text).split()
    normalized_tokens = [normalization_dict.get(token, token) for token in tokens]
    
    return " ".join(normalized_tokens)


print("Kamus normalisasi sederhana berhasil dibuat.")
print("Jumlah kata dalam kamus normalisasi:", len(normalization_dict))

Kamus normalisasi sederhana berhasil dibuat.
Jumlah kata dalam kamus normalisasi: 35


In [15]:
# ============================================================
# Fungsi Cleaning Teks
# ============================================================

def remove_emoji_and_symbols(text):
    """
    Menghapus emoji dan simbol tertentu secara aman menggunakan kategori Unicode.
    Emoji umumnya masuk kategori 'So' atau 'Sk'.
    """
    if pd.isna(text):
        return ""
    
    cleaned_chars = []
    
    for char in str(text):
        unicode_category = unicodedata.category(char)
        
        # So = Symbol, Other
        # Sk = Symbol, Modifier
        if unicode_category in ["So", "Sk"]:
            continue
        
        cleaned_chars.append(char)
    
    return "".join(cleaned_chars)


def clean_youtube_comment(text):
    """
    Fungsi utama untuk membersihkan teks komentar YouTube.
    """
    if pd.isna(text):
        return ""
    
    # Konversi ke string
    text = str(text)
    
    # Membersihkan HTML entity, misalnya &amp; menjadi &
    text = html.unescape(text)
    
    # Normalisasi unicode agar bentuk karakter lebih konsisten
    text = unicodedata.normalize("NFKC", text)
    
    # Lowercase
    text = text.lower()
    
    # Menghapus URL
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    
    # Menghapus mention, misalnya @username
    text = re.sub(r"@[\w\.-]+", " ", text)
    
    # Menghapus hashtag, misalnya #rupiah
    text = re.sub(r"#\w+", " ", text)
    
    # Menghapus newline, tab, dan carriage return
    text = re.sub(r"[\n\r\t]+", " ", text)
    
    # Menghapus emoji dan simbol tertentu
    text = remove_emoji_and_symbols(text)
    
    # Menghapus karakter non-alfanumerik yang tidak relevan
    # Pola ini masih mempertahankan huruf, angka, dan spasi
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    
    # Menghapus underscore agar tidak dianggap sebagai bagian kata
    text = text.replace("_", " ")
    
    # Menghapus spasi berlebih
    text = re.sub(r"\s+", " ", text).strip()
    
    # Normalisasi kata tidak baku sederhana
    text = normalize_slang_words(text)
    
    # Menghapus kembali spasi berlebih setelah normalisasi
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


print("Fungsi cleaning teks berhasil dibuat.")

Fungsi cleaning teks berhasil dibuat.


In [16]:
# ============================================================
# Uji Fungsi Cleaning pada Sampel Aman
# ============================================================

sample_texts = df_work["text_original"].head(5).tolist()

sample_cleaning_result = pd.DataFrame({
    "text_original_preview": [str(text)[:120] for text in sample_texts],
    "text_clean_preview": [clean_youtube_comment(text)[:120] for text in sample_texts]
})

sample_cleaning_result

,text_original_preview,text_clean_preview
0,"gue kira di upload 10 jam yg lalu,ternyata 10 tahun yg lalu😂",saya kira di upload 10 jam yang lalu ternyata 10 tahun yang lalu
1,"Timing nya gabisa lebih pas dari ini😭🙏\nNyentuh Rp.25.000,00 Masih Aman Indonesia😎👌",timing nya gabisa lebih pas dari ini nyentuh rupiah 25 000 00 masih aman indonesia
2,Algoritma YouTube cukup mengerikan 🥀,algoritma youtube cukup mengerikan
3,dyeem 10 year ago lewat beranda kuhh pas bgett;(((,dyeem 10 year ago lewat beranda kuhh pas bgett
4,Algoritma YouTube lebih mengerikan daripada Pria Solo☠️,algoritma youtube lebih mengerikan daripada pria solo


## 6. Cleaning Bertahap dan Audit Jumlah Data

Pada bagian ini dilakukan proses cleaning secara bertahap agar setiap pengurangan jumlah data dapat dilacak.

Tahapan yang dilakukan:

1. Menghapus komentar kosong pada `text_original`.
2. Menghapus duplikasi berdasarkan `comment_id`.
3. Menghapus duplikasi berdasarkan `text_original`.
4. Membuat kolom `text_clean`.
5. Menghapus komentar yang menjadi kosong setelah cleaning.
6. Menghapus komentar yang terlalu pendek setelah cleaning.

Audit jumlah data diperlukan agar proses preprocessing tetap transparan dan dapat dipertanggungjawabkan secara akademik.

In [17]:
# ============================================================
# Proses Cleaning Bertahap
# ============================================================

df_clean = df_work.copy(deep=True)

cleaning_steps = []


def record_cleaning_step(step_name, rows_before, rows_after, description):
    """
    Mencatat perubahan jumlah baris pada setiap tahap cleaning.
    """
    cleaning_steps.append({
        "step_name": step_name,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_removed": rows_before - rows_after,
        "description": description
    })


# ------------------------------------------------------------
# Step 1: Menghapus komentar kosong pada text_original
# ------------------------------------------------------------
rows_before = len(df_clean)

df_clean = df_clean[df_clean["text_original"].notna()].copy()
df_clean["text_original_stripped"] = df_clean["text_original"].astype(str).str.strip()
df_clean = df_clean[df_clean["text_original_stripped"] != ""].copy()
df_clean = df_clean.drop(columns=["text_original_stripped"])

rows_after = len(df_clean)

record_cleaning_step(
    step_name="remove_empty_text_original",
    rows_before=rows_before,
    rows_after=rows_after,
    description="Menghapus baris dengan text_original kosong atau null."
)


# ------------------------------------------------------------
# Step 2: Menghapus duplikasi berdasarkan comment_id
# ------------------------------------------------------------
rows_before = len(df_clean)

if "comment_id" in df_clean.columns:
    df_clean = df_clean.drop_duplicates(subset=["comment_id"], keep="first").copy()

rows_after = len(df_clean)

record_cleaning_step(
    step_name="remove_duplicate_comment_id",
    rows_before=rows_before,
    rows_after=rows_after,
    description="Menghapus duplikasi berdasarkan comment_id."
)


# ------------------------------------------------------------
# Step 3: Menghapus duplikasi berdasarkan text_original
# ------------------------------------------------------------
rows_before = len(df_clean)

df_clean["_text_original_for_duplicate"] = df_clean["text_original"].astype(str).str.strip()
df_clean = df_clean.drop_duplicates(subset=["_text_original_for_duplicate"], keep="first").copy()
df_clean = df_clean.drop(columns=["_text_original_for_duplicate"])

rows_after = len(df_clean)

record_cleaning_step(
    step_name="remove_duplicate_text_original",
    rows_before=rows_before,
    rows_after=rows_after,
    description="Menghapus duplikasi berdasarkan isi text_original."
)


# ------------------------------------------------------------
# Step 4: Membuat kolom statistik sebelum cleaning
# ------------------------------------------------------------
df_clean["text_original_char_count"] = df_clean["text_original"].apply(count_characters)
df_clean["text_original_word_count"] = df_clean["text_original"].apply(count_words)


# ------------------------------------------------------------
# Step 5: Membuat kolom text_clean
# ------------------------------------------------------------
df_clean["text_clean"] = df_clean["text_original"].apply(clean_youtube_comment)


# ------------------------------------------------------------
# Step 6: Membuat kolom statistik setelah cleaning
# ------------------------------------------------------------
df_clean["text_clean_char_count"] = df_clean["text_clean"].apply(count_characters)
df_clean["text_clean_word_count"] = df_clean["text_clean"].apply(count_words)
df_clean["char_count_reduction"] = df_clean["text_original_char_count"] - df_clean["text_clean_char_count"]
df_clean["word_count_reduction"] = df_clean["text_original_word_count"] - df_clean["text_clean_word_count"]


# ------------------------------------------------------------
# Step 7: Menghapus komentar yang kosong setelah cleaning
# ------------------------------------------------------------
rows_before = len(df_clean)

df_clean = df_clean[df_clean["text_clean"].notna()].copy()
df_clean["text_clean_stripped"] = df_clean["text_clean"].astype(str).str.strip()
df_clean = df_clean[df_clean["text_clean_stripped"] != ""].copy()
df_clean = df_clean.drop(columns=["text_clean_stripped"])

rows_after = len(df_clean)

record_cleaning_step(
    step_name="remove_empty_text_clean",
    rows_before=rows_before,
    rows_after=rows_after,
    description="Menghapus komentar yang menjadi kosong setelah proses cleaning."
)


# ------------------------------------------------------------
# Step 8: Menghapus komentar terlalu pendek
# ------------------------------------------------------------
MIN_CLEAN_CHAR_COUNT = 5
MIN_CLEAN_WORD_COUNT = 1

rows_before = len(df_clean)

df_clean = df_clean[
    (df_clean["text_clean_char_count"] >= MIN_CLEAN_CHAR_COUNT) &
    (df_clean["text_clean_word_count"] >= MIN_CLEAN_WORD_COUNT)
].copy()

rows_after = len(df_clean)

record_cleaning_step(
    step_name="remove_too_short_text_clean",
    rows_before=rows_before,
    rows_after=rows_after,
    description=(
        f"Menghapus komentar dengan text_clean kurang dari "
        f"{MIN_CLEAN_CHAR_COUNT} karakter atau kurang dari "
        f"{MIN_CLEAN_WORD_COUNT} kata."
    )
)


# Membuat DataFrame ringkasan cleaning
cleaning_summary_df = pd.DataFrame(cleaning_steps)

print("Proses cleaning bertahap selesai.")
print("Jumlah baris awal df_work:", len(df_work))
print("Jumlah baris akhir df_clean:", len(df_clean))
print("Total baris terhapus:", len(df_work) - len(df_clean))

cleaning_summary_df

Proses cleaning bertahap selesai.
Jumlah baris awal df_work: 3742
Jumlah baris akhir df_clean: 3670
Total baris terhapus: 72


,step_name,rows_before,rows_after,rows_removed,description
0,remove_empty_text_original,3742,3742,0,Menghapus baris dengan text_original kosong atau null.
1,remove_duplicate_comment_id,3742,3742,0,Menghapus duplikasi berdasarkan comment_id.
2,remove_duplicate_text_original,3742,3699,43,Menghapus duplikasi berdasarkan isi text_original.
3,remove_empty_text_clean,3699,3680,19,Menghapus komentar yang menjadi kosong setelah proses cleaning.
4,remove_too_short_text_clean,3680,3670,10,Menghapus komentar dengan text_clean kurang dari 5 karakter atau kurang dari 1 kata.


In [18]:
# ============================================================
# Preview Aman Hasil Cleaning
# ============================================================

preview_columns = []

for col in [
    "comment_id",
    "video_id",
    "published_at",
    "like_count",
    "reply_count",
    "text_original",
    "text_clean",
    "text_original_word_count",
    "text_clean_word_count"
]:
    if col in df_clean.columns:
        preview_columns.append(col)

df_clean_preview = df_clean[preview_columns].copy()

# Membatasi panjang tampilan teks agar tidak terlalu panjang di notebook
df_clean_preview["text_original"] = df_clean_preview["text_original"].astype(str).str.slice(0, 120)
df_clean_preview["text_clean"] = df_clean_preview["text_clean"].astype(str).str.slice(0, 120)

df_clean_preview.head(10)

,comment_id,video_id,published_at,like_count,reply_count,text_original,text_clean,text_original_word_count,text_clean_word_count
0,Ugzo_OAorVrtNPFL3cF4AaABAg,sB7fSPbYFJg,2026-05-24 01:37:15+00:00,560,11,"gue kira di upload 10 jam yg lalu,ternyata 10 tahun yg lalu😂",saya kira di upload 10 jam yang lalu ternyata 10 tahun yang lalu,12,13
1,Ugx2WJ9HFPUNRPaOxON4AaABAg,sB7fSPbYFJg,2026-05-22 13:31:00+00:00,570,12,"Timing nya gabisa lebih pas dari ini😭🙏\nNyentuh Rp.25.000,00 Masih Aman Indonesia😎👌",timing nya gabisa lebih pas dari ini nyentuh rupiah 25 000 00 masih aman indonesia,12,15
2,UgwyjT6JiPsbjsMoUqR4AaABAg,sB7fSPbYFJg,2026-05-24 09:42:48+00:00,271,3,Algoritma YouTube cukup mengerikan 🥀,algoritma youtube cukup mengerikan,5,4
3,UgwN7TGK6eU5papwnNZ4AaABAg,sB7fSPbYFJg,2026-05-21 06:57:48+00:00,321,2,dyeem 10 year ago lewat beranda kuhh pas bgett;(((,dyeem 10 year ago lewat beranda kuhh pas bgett,9,9
4,UgzSaWu-A-vsRbjSc6B4AaABAg,sB7fSPbYFJg,2026-05-23 17:47:36+00:00,92,1,Algoritma YouTube lebih mengerikan daripada Pria Solo☠️,algoritma youtube lebih mengerikan daripada pria solo,7,7
5,Ugy-znLWRvtydcbEUtB4AaABAg,sB7fSPbYFJg,2025-04-04 19:21:36+00:00,390,39,"Hai saya dari masa depan, dan sekarang 1 dolar 17.000😊",hai saya dari masa depan dan sekarang 1 dolar 17 000,10,11
6,UgyS3m8gw1N9OmAMAOZ4AaABAg,sB7fSPbYFJg,2026-05-22 13:05:42+00:00,73,0,Sepertinya ini gak kebetulan lewat nya 🗿,sepertinya ini tidak kebetulan lewat nya,7,6
7,UgxLk7UumzTZAaoHOT94AaABAg,sB7fSPbYFJg,2026-05-22 13:05:08+00:00,63,0,Bro had a vision 🗿,bro had a vision,5,4
8,UgytHQ_zOn6DAcRKUOd4AaABAg,sB7fSPbYFJg,2026-05-25 08:54:03+00:00,14,0,algoritma youtube lebih baik daripada rupiah kita saat ini dawg🥀😭,algoritma youtube lebih baik daripada rupiah kita saat ini dawg,10,10
9,Ugyg-zgbcdaIVrYoDfh4AaABAg,sB7fSPbYFJg,2026-05-17 10:30:09+00:00,121,8,"2026 ini ada mbg saya curiga rupiah dicetak trus sampai nilainya merendah, singkat nya semakin mudah barang itu dite...",2026 ini ada mbg saya curiga rupiah dicetak terus sampai nilainya merendah singkat nya semakin mudah barang itu dite...,23,23


In [19]:
# ============================================================
# Ringkasan Perbandingan Sebelum dan Sesudah Cleaning
# ============================================================

cleaning_overview_df = pd.DataFrame({
    "metric": [
        "source_file",
        "rows_before_cleaning",
        "rows_after_cleaning",
        "rows_removed_total",
        "rows_removed_percentage",
        "columns_before_cleaning",
        "columns_after_cleaning",
        "avg_text_original_char_count",
        "avg_text_clean_char_count",
        "avg_text_original_word_count",
        "avg_text_clean_word_count",
        "min_clean_char_threshold",
        "min_clean_word_threshold"
    ],
    "value": [
        latest_raw_file.name,
        len(df_work),
        len(df_clean),
        len(df_work) - len(df_clean),
        round(((len(df_work) - len(df_clean)) / len(df_work)) * 100, 2) if len(df_work) > 0 else 0,
        df_work.shape[1],
        df_clean.shape[1],
        round(df_clean["text_original_char_count"].mean(), 2),
        round(df_clean["text_clean_char_count"].mean(), 2),
        round(df_clean["text_original_word_count"].mean(), 2),
        round(df_clean["text_clean_word_count"].mean(), 2),
        MIN_CLEAN_CHAR_COUNT,
        MIN_CLEAN_WORD_COUNT
    ]
})

cleaning_overview_df

,metric,value
0,source_file,youtube_comments_raw_20260529_141537.csv
1,rows_before_cleaning,3742
2,rows_after_cleaning,3670
3,rows_removed_total,72
4,rows_removed_percentage,1.92
5,columns_before_cleaning,13
6,columns_after_cleaning,20
7,avg_text_original_char_count,110.3
8,avg_text_clean_char_count,107.79
9,avg_text_original_word_count,16.98


## 7. Menyimpan Dataset Cleaned dan Laporan Cleaning

Setelah proses cleaning selesai, hasilnya perlu disimpan agar dapat digunakan pada tahap berikutnya.

Pada bagian ini akan disimpan tiga output utama:

1. Dataset cleaned ke folder `data/processed/`.
2. Ringkasan tahapan cleaning ke folder `reports/`.
3. Ringkasan overview cleaning ke folder `reports/`.

Dataset cleaned tetap perlu diperlakukan secara hati-hati karena masih memuat teks komentar asli dan teks hasil cleaning. Oleh karena itu, file pada folder `data/processed/` tidak langsung direkomendasikan untuk dipublikasikan ke GitHub sebelum dilakukan pemeriksaan kembali.

File laporan pada folder `reports/` lebih aman untuk dipublikasikan karena hanya berisi ringkasan jumlah data, bukan isi komentar atau identitas author.

In [20]:
# ============================================================
# Validasi Keamanan Kolom Sebelum Dataset Cleaned Disimpan
# ============================================================

sensitive_columns = ["author", "author_name", "channel_author", "user_name", "username"]

existing_sensitive_columns = [
    col for col in sensitive_columns if col in df_clean.columns
]

if existing_sensitive_columns:
    print("PERINGATAN: Masih ditemukan kolom sensitif berikut:")
    print(existing_sensitive_columns)
else:
    print("Validasi aman: tidak ditemukan kolom author atau kolom sensitif sejenis pada df_clean.")

print("\nJumlah baris df_clean:", df_clean.shape[0])
print("Jumlah kolom df_clean:", df_clean.shape[1])

print("\nDaftar kolom df_clean:")
print(df_clean.columns.tolist())

Validasi aman: tidak ditemukan kolom author atau kolom sensitif sejenis pada df_clean.

Jumlah baris df_clean: 3670
Jumlah kolom df_clean: 20

Daftar kolom df_clean:
['video_id', 'video_title', 'video_url', 'channel_title', 'video_published_at', 'comment_id', 'published_at', 'comment_updated_at', 'text_original', 'like_count', 'reply_count', 'crawl_timestamp', 'source_file', 'text_original_char_count', 'text_original_word_count', 'text_clean', 'text_clean_char_count', 'text_clean_word_count', 'char_count_reduction', 'word_count_reduction']


In [21]:
# ============================================================
# Menyimpan Dataset Cleaned dan Laporan Cleaning
# ============================================================

# Timestamp agar file output memiliki versi yang jelas
output_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Nama file output
cleaned_dataset_filename = f"youtube_comments_cleaned_{output_timestamp}.csv"
cleaning_summary_filename = f"cleaning_summary_{output_timestamp}.csv"
cleaning_overview_filename = f"cleaning_overview_{output_timestamp}.csv"

# Path output
cleaned_dataset_path = processed_data_dir / cleaned_dataset_filename
cleaning_summary_path = reports_dir / cleaning_summary_filename
cleaning_overview_path = reports_dir / cleaning_overview_filename

# Simpan file
df_clean.to_csv(cleaned_dataset_path, index=False, encoding="utf-8-sig")
cleaning_summary_df.to_csv(cleaning_summary_path, index=False, encoding="utf-8-sig")
cleaning_overview_df.to_csv(cleaning_overview_path, index=False, encoding="utf-8-sig")

print("File berhasil disimpan.")

print("\nDataset cleaned:")
print(cleaned_dataset_path)

print("\nCleaning summary:")
print(cleaning_summary_path)

print("\nCleaning overview:")
print(cleaning_overview_path)

File berhasil disimpan.

Dataset cleaned:
D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis\data\processed\youtube_comments_cleaned_20260529_150114.csv

Cleaning summary:
D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis\reports\cleaning_summary_20260529_150114.csv

Cleaning overview:
D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis\reports\cleaning_overview_20260529_150114.csv


In [22]:
# ============================================================
# Verifikasi File Output
# ============================================================

output_files_check = pd.DataFrame({
    "file_type": [
        "cleaned_dataset",
        "cleaning_summary",
        "cleaning_overview"
    ],
    "file_path": [
        str(cleaned_dataset_path),
        str(cleaning_summary_path),
        str(cleaning_overview_path)
    ],
    "file_exists": [
        cleaned_dataset_path.exists(),
        cleaning_summary_path.exists(),
        cleaning_overview_path.exists()
    ],
    "file_size_kb": [
        round(cleaned_dataset_path.stat().st_size / 1024, 2) if cleaned_dataset_path.exists() else 0,
        round(cleaning_summary_path.stat().st_size / 1024, 2) if cleaning_summary_path.exists() else 0,
        round(cleaning_overview_path.stat().st_size / 1024, 2) if cleaning_overview_path.exists() else 0
    ]
})

output_files_check

,file_type,file_path,file_exists,file_size_kb
0,cleaned_dataset,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis\data\processed\youtube_comments_cleane...,True,1929.50
1,cleaning_summary,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis\reports\cleaning_summary_20260529_1501...,True,0.55
2,cleaning_overview,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis\reports\cleaning_overview_20260529_150...,True,0.42


In [23]:
# ============================================================
# Membaca Ulang Dataset Cleaned untuk Validasi
# ============================================================

df_cleaned_check = pd.read_csv(cleaned_dataset_path)

print("Dataset cleaned berhasil dibaca ulang.")
print("Jumlah baris:", df_cleaned_check.shape[0])
print("Jumlah kolom:", df_cleaned_check.shape[1])

print("\nApakah jumlah baris sesuai df_clean?")
print(df_cleaned_check.shape[0] == df_clean.shape[0])

print("\nApakah jumlah kolom sesuai df_clean?")
print(df_cleaned_check.shape[1] == df_clean.shape[1])

Dataset cleaned berhasil dibaca ulang.
Jumlah baris: 3670
Jumlah kolom: 20

Apakah jumlah baris sesuai df_clean?
True

Apakah jumlah kolom sesuai df_clean?
True


## 8. Interpretasi Hasil Cleaning

Berdasarkan proses cleaning yang telah dilakukan, jumlah data awal sebanyak 3.742 baris berkurang menjadi 3.670 baris. Total data yang terhapus adalah 72 baris atau sekitar 1,92% dari keseluruhan data.

Pengurangan data tersebut masih tergolong wajar karena proses cleaning hanya menghapus data yang kosong, duplikat, atau terlalu pendek setelah dibersihkan. Dengan demikian, proses preprocessing tidak bersifat terlalu agresif dan masih mempertahankan sebagian besar informasi komentar.

Rata-rata jumlah karakter sebelum cleaning adalah sekitar 110,3 karakter, sedangkan setelah cleaning menjadi sekitar 107,79 karakter. Penurunan ini menunjukkan bahwa proses cleaning berhasil menghapus elemen yang kurang relevan seperti simbol, emoji, URL, mention, hashtag, newline, tab, dan spasi berlebih.

Rata-rata jumlah kata setelah cleaning sedikit meningkat dari 16,98 menjadi 17,05. Hal ini dapat terjadi karena proses normalisasi kata tidak baku, misalnya kata singkat atau bentuk informal tertentu diubah menjadi bentuk kata yang lebih standar.

Secara keseluruhan, dataset hasil cleaning sudah lebih siap digunakan untuk tahap berikutnya, yaitu labeling sentimen. Namun, pada notebook ini belum dilakukan labeling, stemming, stopword removal, TF-IDF, maupun modeling agar ruang lingkup tahap preprocessing tetap terjaga.

In [24]:
# ============================================================
# Preview Final Aman Dataset Cleaned
# ============================================================

final_preview_columns = [
    col for col in [
        "comment_id",
        "video_id",
        "published_at",
        "like_count",
        "reply_count",
        "text_original",
        "text_clean",
        "text_original_char_count",
        "text_clean_char_count",
        "text_original_word_count",
        "text_clean_word_count"
    ]
    if col in df_cleaned_check.columns
]

df_final_preview = df_cleaned_check[final_preview_columns].copy()

# Membatasi panjang teks agar preview tetap aman dan ringkas
if "text_original" in df_final_preview.columns:
    df_final_preview["text_original"] = df_final_preview["text_original"].astype(str).str.slice(0, 100)

if "text_clean" in df_final_preview.columns:
    df_final_preview["text_clean"] = df_final_preview["text_clean"].astype(str).str.slice(0, 100)

df_final_preview.head(10)

,comment_id,video_id,published_at,like_count,reply_count,text_original,text_clean,text_original_char_count,text_clean_char_count,text_original_word_count,text_clean_word_count
0,Ugzo_OAorVrtNPFL3cF4AaABAg,sB7fSPbYFJg,2026-05-24 01:37:15+00:00,560,11,"gue kira di upload 10 jam yg lalu,ternyata 10 tahun yg lalu😂",saya kira di upload 10 jam yang lalu ternyata 10 tahun yang lalu,60,64,12,13
1,Ugx2WJ9HFPUNRPaOxON4AaABAg,sB7fSPbYFJg,2026-05-22 13:31:00+00:00,570,12,"Timing nya gabisa lebih pas dari ini😭🙏\nNyentuh Rp.25.000,00 Masih Aman Indonesia😎👌",timing nya gabisa lebih pas dari ini nyentuh rupiah 25 000 00 masih aman indonesia,82,82,12,15
2,UgwyjT6JiPsbjsMoUqR4AaABAg,sB7fSPbYFJg,2026-05-24 09:42:48+00:00,271,3,Algoritma YouTube cukup mengerikan 🥀,algoritma youtube cukup mengerikan,36,34,5,4
3,UgwN7TGK6eU5papwnNZ4AaABAg,sB7fSPbYFJg,2026-05-21 06:57:48+00:00,321,2,dyeem 10 year ago lewat beranda kuhh pas bgett;(((,dyeem 10 year ago lewat beranda kuhh pas bgett,50,46,9,9
4,UgzSaWu-A-vsRbjSc6B4AaABAg,sB7fSPbYFJg,2026-05-23 17:47:36+00:00,92,1,Algoritma YouTube lebih mengerikan daripada Pria Solo☠️,algoritma youtube lebih mengerikan daripada pria solo,55,53,7,7
5,Ugy-znLWRvtydcbEUtB4AaABAg,sB7fSPbYFJg,2025-04-04 19:21:36+00:00,390,39,"Hai saya dari masa depan, dan sekarang 1 dolar 17.000😊",hai saya dari masa depan dan sekarang 1 dolar 17 000,54,52,10,11
6,UgyS3m8gw1N9OmAMAOZ4AaABAg,sB7fSPbYFJg,2026-05-22 13:05:42+00:00,73,0,Sepertinya ini gak kebetulan lewat nya 🗿,sepertinya ini tidak kebetulan lewat nya,40,40,7,6
7,UgxLk7UumzTZAaoHOT94AaABAg,sB7fSPbYFJg,2026-05-22 13:05:08+00:00,63,0,Bro had a vision 🗿,bro had a vision,18,16,5,4
8,UgytHQ_zOn6DAcRKUOd4AaABAg,sB7fSPbYFJg,2026-05-25 08:54:03+00:00,14,0,algoritma youtube lebih baik daripada rupiah kita saat ini dawg🥀😭,algoritma youtube lebih baik daripada rupiah kita saat ini dawg,65,63,10,10
9,Ugyg-zgbcdaIVrYoDfh4AaABAg,sB7fSPbYFJg,2026-05-17 10:30:09+00:00,121,8,"2026 ini ada mbg saya curiga rupiah dicetak trus sampai nilainya merendah, singkat nya semakin mudah",2026 ini ada mbg saya curiga rupiah dicetak terus sampai nilainya merendah singkat nya semakin mudah,143,143,23,23
